# CPU

In [20]:
import os

In [21]:
print(os.cpu_count())

12


In [22]:
import numpy as np

# tifffile loading measurement

In [24]:
from pytorch.utils.util import load_hs_tiff
import time

In [25]:
def measure_time_load(func, arg):
    print("-" * 40)
    
    start_time = time.perf_counter()
    
    func(arg)
    
    end_time = time.perf_counter()
    
    duration = end_time - start_time
    
    print(f"Time taken: {duration:.6f} seconds")

def measure_time_load_list(func, arg_list):
    print("-" * 40)
    
    start_time = time.perf_counter()
    
    func(*arg_list)
    
    end_time = time.perf_counter()
    
    duration = end_time - start_time
    
    print(f"Time taken: {duration:.6f} seconds")

In [26]:
measure_time_load(load_hs_tiff,"data\\data_raw\\hyperleaf\\00000.tiff")

----------------------------------------
Time taken: 0.746832 seconds


In [27]:
np.save('0000.npy', load_hs_tiff("data\\data_raw\\hyperleaf\\00000.tiff"))

In [28]:
measure_time_load(np.load,"0000.npy")

----------------------------------------
Time taken: 0.005405 seconds


# Normalization measurement time

In [29]:
from pytorch.utils.util import load_pkl

In [30]:
preprocess_dict = load_pkl('preprocess')

In [31]:
data = load_hs_tiff("data\\data_raw\\hyperleaf\\00000.tiff")

In [32]:
def normalize(array: np.ndarray, mean_list, std_list):
    C, _, _ = array.shape
    mean_arr = np.array(mean_list).reshape(C, 1, 1)
    std_arr = np.array(std_list).reshape(C, 1, 1)
    array_float = array.astype(np.float32)
    
    normalized_array = (array_float - mean_arr) / std_arr

In [33]:
measure_time_load_list(normalize,[data, preprocess_dict['mean'], preprocess_dict['std']])

----------------------------------------
Time taken: 0.018488 seconds


# Torch normalization measurement time

In [34]:
import torch
from torchvision.transforms import v2

In [35]:
transform_func = v2.Compose([
    #v2.RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.333)),
    #v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32),
    v2.Normalize(mean=preprocess_dict['mean'], std=preprocess_dict['std']),
])

In [36]:
def normalize_torch(array: torch.Tensor):
    transform_func(array)

In [37]:
measure_time_load(normalize_torch,torch.from_numpy(data))

----------------------------------------
Time taken: 0.093131 seconds


In [38]:
measure_time_load(normalize_torch,torch.Tensor(data))

----------------------------------------
Time taken: 0.002339 seconds


In [39]:
torch.from_numpy(data).max()

tensor(64963.)

In [40]:
transform_func(torch.from_numpy(data)).max()

tensor(11.8590)

In [41]:
torch.from_numpy(data).shape

torch.Size([204, 48, 352])

# Checking numpy memmap time

In [42]:
import random

In [43]:
start_time = time.perf_counter()

data_mmap = np.load("0000.npy", mmap_mode='r+')

end_time = time.perf_counter()

duration = end_time - start_time

print(f"Time taken: {duration:.6f} seconds")

####################################

start_time = time.perf_counter()

for i in range(1000):

    ind1 = random.randint(0, 203)

    ind2 = random.randint(0, 47)

    ind3 = random.randint(0, 351)

    data_mmap[ind1, ind2, ind3]

end_time = time.perf_counter()

duration = end_time - start_time

print(f"Time taken: {duration:.6f} seconds")

Time taken: 0.000960 seconds
Time taken: 0.004269 seconds


In [44]:
start_time = time.perf_counter()

data_mmap = np.load("0000.npy")

end_time = time.perf_counter()

duration = end_time - start_time

print(f"Time taken: {duration:.6f} seconds")

####################################

start_time = time.perf_counter()

for i in range(1000):

    ind1 = random.randint(0, 203)

    ind2 = random.randint(0, 47)

    ind3 = random.randint(0, 351)

    data_mmap[ind1, ind2, ind3]

end_time = time.perf_counter()

duration = end_time - start_time

print(f"Time taken: {duration:.6f} seconds")

Time taken: 0.005290 seconds
Time taken: 0.002064 seconds


# Pandas

In [45]:
import pandas as pd

In [46]:
start_time = time.perf_counter()

df = pd.read_csv("data/data_csv/hyperleaf/train.csv", dtype={"ImageId": str})

end_time = time.perf_counter()

duration = end_time - start_time

print(f"Time taken: {duration:.6f} seconds")

Time taken: 0.007101 seconds


In [47]:
start_time = time.perf_counter()

img_path = os.path.join("", str(df.iloc[1500, 0]) + ".npy")

end_time = time.perf_counter()

duration = end_time - start_time

print(f"Time taken: {duration:.6f} seconds")

Time taken: 0.000233 seconds


# Pickle

In [48]:
from pytorch.utils.util import load_pkl

In [49]:
start_time = time.perf_counter()

load_pkl("preprocess")

end_time = time.perf_counter()

duration = end_time - start_time

print(f"Time taken: {duration:.6f} seconds")

Time taken: 0.000355 seconds


# DataLoader

In [50]:
loader = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] 
train_iter = iter(loader)  
next_batch = next(train_iter)

for i in range(len(loader)):
    batch = next_batch
    print(i, batch)
    if i + 1 != len(loader):
        next_batch = next(train_iter)

0 0
1 1
2 2
3 3
4 4
5 5
6 6
7 7
8 8
9 9


# Загрузка датасета

In [51]:
import pandas as pd

In [52]:
train_csv = pd.read_csv("data/data_csv/hyperleaf/train.csv")
test_csv = pd.read_csv("data/data_csv/hyperleaf/test.csv")
sol_csv = pd.read_csv("data/data_csv/hyperleaf/solution.csv")

In [53]:
train_csv.head()

,ImageId,GrainWeight,Gsw,PhiPS2,Fertilizer,Heerup,Kvium,Rembrandt,Sheriff
0,0,4004,0.067527,0.627443,0.0,0,1,0,0
1,1,6753,0.062773,0.664247,0.5,0,0,0,1
2,3,8082,0.123482,0.425904,1.0,0,0,1,0
3,7,8520,0.007312,0.691565,1.0,0,0,0,1
4,9,8520,0.129374,0.502281,1.0,0,0,0,1


In [54]:
train_csv.iloc[0, 5]

np.int64(0)

In [55]:
last_4_cols = df.iloc[:, -4:]
print(last_4_cols)

      Heerup  Kvium  Rembrandt  Sheriff
0          0      1          0        0
1          0      0          0        1
2          0      0          1        0
3          0      0          0        1
4          0      0          0        1
...      ...    ...        ...      ...
1585       0      1          0        0
1586       0      0          1        0
1587       0      1          0        0
1588       0      1          0        0
1589       0      1          0        0

[1590 rows x 4 columns]


In [56]:
df_series_all = last_4_cols.values
print(df_series_all)

[[0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]
 ...
 [0 1 0 0]
 [0 1 0 0]
 [0 1 0 0]]


In [57]:
spec_col = df.iloc[2, -4:].values.astype(int)
print(spec_col)

[0 0 1 0]


In [58]:
df_series_max = last_4_cols.values.argmax(axis=1)
print(df_series_max)

[1 3 2 ... 1 1 1]


In [59]:
columns = ['Heerup',	'Kvium',	'Rembrandt',	'Sheriff']

In [60]:
for column in columns:
    print(column, train_csv[(train_csv[column]==1)].shape[0])

Heerup 410
Kvium 350
Rembrandt 430
Sheriff 400


In [61]:
for column in columns:
    print(column, sol_csv[(sol_csv[column]==1)].shape[0])

Heerup 220
Kvium 270
Rembrandt 130
Sheriff 200


In [62]:
for column_1 in columns:
    for column_2 in columns:
        if column_1 != column_2:
            print(column_1, column_2, train_csv[(train_csv[column_1]==1) & (train_csv[column_2]==1)].shape[0])

Heerup Kvium 0
Heerup Rembrandt 0
Heerup Sheriff 0
Kvium Heerup 0
Kvium Rembrandt 0
Kvium Sheriff 0
Rembrandt Heerup 0
Rembrandt Kvium 0
Rembrandt Sheriff 0
Sheriff Heerup 0
Sheriff Kvium 0
Sheriff Rembrandt 0


In [63]:
for column_1 in columns:
    for column_2 in columns:
        if column_1 != column_2:
            print(column_1, column_2, sol_csv[(sol_csv[column_1]==1) & (sol_csv[column_2]==1)].shape[0])

Heerup Kvium 0
Heerup Rembrandt 0
Heerup Sheriff 0
Kvium Heerup 0
Kvium Rembrandt 0
Kvium Sheriff 0
Rembrandt Heerup 0
Rembrandt Kvium 0
Rembrandt Sheriff 0
Sheriff Heerup 0
Sheriff Kvium 0
Sheriff Rembrandt 0


In [64]:
[0]*len([1,2,3])

[0, 0, 0]

In [71]:
[x.item() for x in torch.linspace(0, 0.15, sum(sum([[1,1], [1,1], [1,1]], [])))]

[0.0,
 0.02500000037252903,
 0.05000000074505806,
 0.07500000298023224,
 0.10000000894069672,
 0.125,
 0.15000000596046448]

# Torch stuff

In [3]:
import torch

In [20]:
x = torch.Tensor([[[[1,5], [1,1]], [[1,1], [8,1]]], [[[1,1], [30,1]], [[1,1], [1,100]]]])

In [21]:
x.shape

torch.Size([2, 2, 2, 2])

In [22]:
x = torch.functional.F.adaptive_avg_pool2d(x, output_size=1)

In [23]:
x

tensor([[[[ 2.0000]],

         [[ 2.7500]]],


        [[[ 8.2500]],

         [[25.7500]]]])

In [17]:
x.shape

torch.Size([2, 2, 1, 1])

In [ ]:
x.view(x.size(0), -1)

torch.Size([2, 2])

In [ ]:
x = x.view(x.size(0), -1)

# Arguments

In [1]:
import argparse

In [ ]:
parser = argparse.ArgumentParser('FCMAE fine-tuning', add_help=False)


parser.add_argument('--update_freq', default=1, type=int,
                    help='gradient accumulation steps')

# EMA related parameters
parser.add_argument('--model_ema', type=str2bool, default=False)
parser.add_argument('--model_ema_decay', type=float, default=0.9999, help='')
parser.add_argument('--model_ema_force_cpu', type=str2bool, default=False, help='')
parser.add_argument('--model_ema_eval', type=str2bool, default=False, help='Using ema to eval during training.')

# Optimization parameters
parser.add_argument('--clip_grad', type=float, default=None, metavar='NORM',
                    help='Clip gradient norm (default: None, no clipping)')
parser.add_argument('--lr', type=float, default=None, metavar='LR',
                    help='learning rate (absolute lr)')
parser.add_argument('--blr', type=float, default=5e-4, metavar='LR',
                    help='base learning rate: absolute_lr = base_lr * total_batch_size / 256')
parser.add_argument('--layer_decay', type=float, default=1.0)
parser.add_argument('--min_lr', type=float, default=1e-6, metavar='LR',
                    help='lower lr bound for cyclic schedulers that hit 0 (1e-6)')
parser.add_argument('--warmup_epochs', type=int, default=20, metavar='N',
                    help='epochs to warmup LR, if scheduler supports')

parser.add_argument('--warmup_steps', type=int, default=-1, metavar='N',
                    help='num of steps to warmup LR, will overload warmup_epochs if set > 0')    
parser.add_argument('--opt', default='adamw', type=str, metavar='OPTIMIZER',
                    help='Optimizer (default: "adamw"')
parser.add_argument('--opt_eps', default=1e-8, type=float, metavar='EPSILON',
                    help='Optimizer Epsilon (default: 1e-8)')
parser.add_argument('--opt_betas', default=None, type=float, nargs='+', metavar='BETA',
                    help='Optimizer Betas (default: None, use opt default)')
parser.add_argument('--momentum', type=float, default=0.9, metavar='M',
                    help='SGD momentum (default: 0.9)')
parser.add_argument('--weight_decay_end', type=float, default=None, help="""Final value of the
    weight decay. We use a cosine schedule for WD and using a larger decay by
    the end of training improves performance for ViTs.""")

parser.add_argument('--train_interpolation', type=str, default='bicubic',
                    help='Training interpolation (random, bilinear, bicubic default: "bicubic")')

# * Finetuning params
parser.add_argument('--head_init_scale', default=0.001, type=float,
                    help='classifier head initial scale, typically adjusted in fine-tuning')

# Evaluation parameters
parser.add_argument('--crop_pct', type=float, default=None)

# distributed training parameters
parser.add_argument('--world_size', default=1, type=int,
                    help='number of distributed processes')
parser.add_argument('--local_rank', default=-1, type=int)
parser.add_argument('--dist_on_itp', type=str2bool, default=False)
parser.add_argument('--dist_url', default='env://',
                    help='url used to set up distributed training')

parser.add_argument('--use_amp', type=str2bool, default=False, 
                    help="Use apex AMP (Automatic Mixed Precision) or not")